In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [ ]:
import yaml

from sim.drive_simulator import CarSim
from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)

# プログラムの書き方講座3

## 1. 条件に応じて動作を変えよう

- プログラムは、特定の条件が成立するかしないかによって、実行する命令を変えることができる
  - これを「条件分岐」と呼ぶ 
- 例えば「虫を見つけたら回転せよ」というプログラムを日本語で書くと
  ```
  虫を見つけよ
  ・もしみつけてないなら
    ・停止せよ
  ・もし見つけたなら
  　・回転せよ
  ```
- 上記をプログラミング言語(python)では下記のように書ける
  ```
  pos = search()
  if pos is None:
    rotate(w=0)
  else:
    rotate(w=90)
  ```

- 解説
  - `if 条件文:` で、「もし、条件文が成り立つなら」という意味になる
    - 最後の:も忘れないように
  - `else:`で「条件文が成り立たないなら」という意味になる
    - `else:`の中に書きたい命令がない場合、`else:`は書かなくてよい
  - 命令は、`if 条件文:`や`else:`の下に段落をつくって（左側にスペースを入れて）かく
  - 条件文は、「値　比較記号　値」の形で書く
    - 比較記号には >, <, >=, <=, ==, !=, is, is not　等が使える
    - 値には「変数」「数値」「文字列」「None」等が使える
    - 条件の例１：　pos is None 
        - 「posがない場合」という条件になる
    - 条件の例２：　pos is not None 
        - 「posがある場合」という条件になる
    - 条件の例３：　pos.theta > 10
        - 「pos.thetaが10より大きい場合」という条件になる
    - 条件の例４：　10 < pos.theta
        - 例３と同じ意味
    - 条件の例５：　pos.theta == 0 
        - 「pos.thetaが0の場合」という条件になる
    - 条件の例６：　pos.name == "bug" 
        - 「pos.nameがbugである場合」という条件になる

## 2. より複雑な条件

- 分岐は多段階にも書ける
  - 例えば「条件文１が成り立たない時に、さらに条件文２が成り立つかどうかで命令を分岐したい」場合は下記のように書ける
    ```
    if 条件文１:
      命令()
      命令()
      ・・・略
    else:
      if 条件文２:
        命令()
        命令()
        ・・・略
      else:
        命令()
        命令()
        ・・・略
    ```
- 組み合わせの条件文「〇〇かつ△△」や「〇〇もしくは△△」等を書きたい場合
  - `条件文1 and 条件文2`と書くと、「条件文1と条件文2の両方が正しい場合に成り立つ」という条件文になる
  - `条件文1 or 条件文2`と書くと、「条件文1もしくは条件文2のうち少なくとも１つは正しい場合に成り立つ」という条件文になる
  - `not 条件文1`と書くと、「条件文1が正しくない場合に成り立つ」という条件文になる
- 例えば、「ほぼ正面に虫を見つけたら、虫の位置まで前進する」というプログラムを日本語で書くと
  ```
  虫を見つけよ
  ・もし見つけたなら
  　・もし虫がほぼ正面（±5度以内）なら
  　　・虫の位置まで前進せよ
  ```
- 上記をプログラミング言語に直すと下記になる
  ```
  pos = search()
  if pos is None:
    move(v=0)
  else:
    if pos.theta <= 5 and pos.theta >= -5:
      move(v=0.2, t=pos.x/0.2)
      wait()
    else:
      move(v=0)
  ```


## 3.プログラムを繰り返そう

- 上記の「ほぼ正面に虫を見つけたら、虫の位置まで前進する」プログラムだと虫を探す処理を１度だけして、その結果に応じて一度だけ処理を行うとプログラムが終了してしまう。「ほぼ正面に虫を見つけたら、虫の位置まで前進する」という処理をずっとし続けたい（いつ虫が来てもいいように）という場合は、下記のように書くことで繰り返し同じプログラムを実行できる。
  - 繰り返したい処理は、段落をつくって（左側にスペースを入れて）かく
  ```
  while True:
    繰り返したい処理
  ```
- 「ほぼ正面に虫を見つけたら、虫の位置まで前進する」をずっとしたい場合は以下のように書ける
  ```
  while True:
    pos = search()
    if pos is None:
      move(v=0)
    else:
      if pos.theta <= 5 and pos.theta >= -5:
        move(v=0.2, t=pos.x/0.2)
        wait()
      else:
        move(v=0)
  ```

# チュートリアル3

下記の命令を組み合わせてプログラムを書き、ロボットを直進させながら標識(標識名はsign1)が車にぶつかりそう（左右0.2[m]以内）に見えた瞬間に停止しよう。

## 取り組み方
1. 使える命令を理解する
    - チュートリアル2と同じだがrotateは使用不可
2. 下のセルを実行して、ロボットの限界速度や、ロボットが存在する初期位置やチェックポイント（goal）を把握する
3. ２つ下のセル内にプログラムを書き実行して結果を見る

|使える命令|意味|指定できる値|使い方|
|--|--|--|--|
|move|一定速度で前に進む|v=速度[m/s]|move(v=0.2)|
|move|一定時間だけ一定速度で前に進む|v=速度[m/s], t=時間[s]|move(v=0.2, t=1.0)|
|wait|直前の命令が終わるまで待つ|-|wait()|
|search|標識を見つける（複数見つかった場合は、最も近いもの）|-|pos = Search()|
|search|特定の標識を見つける（複数見つかった場合は、最も近いもの）|name = "標識名"|pos = Search(name="sign1")|

- pos = search() が返す値には下記が含まれる
  - pos.x: 見つけた標識の前方位置[m]　※前方が正
  - pox.y: 見つけた標識の左右位置[m]　※左側が正、右側は負
  - pos.r: 見つけた標識への距離[m]
  - pos.theta: 見つけた標識の角度[度]　※左側が正、右側は負
  - pos.name: 見つけた標識の標識名

## 注意点
- スタート時の位置はランダムに前後最大30cmほどずれる（左右ずれはない）

In [ ]:
class Tutorial3(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=20)
        self.goals = [
            GoalCircle((2.2, 0.0), 0.2, should_stop=True),
        ]
        self.initial_xy = (0.0, 0.0)
        self.random_d_xy = (0.3, 0.0)
        self.set_signs(
            [
                Sign(x=1.9, y=-0.4, name="sign1"),
                Sign(x=2.7, y=0.4, name="sign1"),
                Sign(x=3.5, y=-0.1, name="sign1"),
            ]
        )


print("最大速度", prop.max_velocity, "m/s")
print("最大回転速度", prop.max_rotate_deg, "度/s")
MissionDrawer(Tutorial3()).show()

In [ ]:
class Tutorial3(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=20)
        self.goals = [
            GoalCircle((2.2, 0.0), 0.2, should_stop=True),
        ]
        self.initial_xy = (0.0, 0.0)
        self.random_d_xy = (0.3, 0.0)
        self.set_signs(
            [
                Sign(x=1.9, y=-0.4, name="sign1"),
                Sign(x=2.7, y=0.4, name="sign1"),
                Sign(x=3.5, y=-0.1, name="sign1"),
            ]
        )

    @staticmethod
    def command_func(*, move, search, wait, **kwargs):
        # ヒントとして、「searchして標識が見つからなかったら0.1[m]だけ前進する」を繰り返すという処理を記載済み
        # elseの中だけプログラムを書けばOK

        while True:
            pos = search(name="sign1")
            if pos is None:
                move(v=0.2, t=0.5)
                wait()
            else:
                ######## ここから下に「標識が左右0.2[m]以内でなければ0.1[m]だけ前進する」プログラムを書こう
                move(v=0.2, t=0.5)  # 記載例
                wait()
                ######## ここより上にプログラムを書こう
        ######## プログラムを書いた後にセルを実行し結果を確認しよう

sim = CarSim(prop,Tutorial3())
success = sim.run()
if success:
    SimDrawer(sim).show()